***

Preparing Workspace

***

Data is downloaded manually from here https://www.huduser.gov/portal/datasets/cp.html#data_2006-2021 using the "Data" tab next to the "Query Tool"

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import itertools
# pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.max_columns', None)

from __future__ import annotations
from typing import Union
import urllib3
import json
from distutils.log import warn


## Setting file paths ---

# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'HUD')
    path_chas = os.path.join(path_users, 'Documents', 'Projects', 'General', 'Regional Monitoring', 'CHAS')
    path_data = os.path.join(path_chas , 'Data')
    path_dict = os.path.join(path_chas , 'Dictionaries')

path_code    = os.path.join(path_git, 'Data', 'HUD')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')



***

Importing

***

In [ ]:
df_codes = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name='CDPcodes')
df_codes = df_codes[df_codes['MPO'] == 'SACOG']
df_codes['place'] = df_codes['place'].astype(str).apply('{:0>5}'.format)
df_codes['state'] = df_codes['state'].astype(str).apply('{:0>2}'.format)

def split_est(x):
    return int(x.split('est')[1])


list_df_years = []
for years in os.listdir(path_data):
    print(''); print(''); print(years)
    list_df_tables = []
    for table in tqdm(os.listdir(os.path.join(path_data, years))):
        df = pd.read_csv(os.path.join(path_data, years, table), encoding='latin-1')
        if 'PLACE' in df.columns:
            df = df.rename(columns={'NAME':'name', 'ST':'st', 'PLACE':'place'})
        df['place'] = df['place'].astype(str).apply('{:0>5}'.format)
        df['st'   ] = df['st'   ].astype(str).apply('{:0>2}'.format)
        df = df.drop(['sumlevel', 'geoid'], axis=1)
        df = df[(df['place'].isin(df_codes['place'].unique())) & (df['st'].isin(df_codes['state'].unique()))]
        df = df.merge(df_codes[['place', 'County Name']], on='place', how='left')
        df.loc[df['place'].isin(df_codes[df_codes['Incorporated'] != 'Yes']['place'].unique()), 'name' ] = 'Unincorporated'
        df.loc[df['place'].isin(df_codes[df_codes['Incorporated'] != 'Yes']['place'].unique()), 'place'] = 'Unincorporated'
        df = df.groupby(['source', 'st', 'County Name', 'place', 'name'], as_index=False).sum()
        df = df.melt(id_vars = ['source', 'st', 'County Name', 'place', 'name'], var_name = 'Estimate', value_name = 'Households')
        df = df[~df['Estimate'].str.contains('moe')] # Need to roll up MOE separately
        df['sort'] = df['Estimate'].apply(split_est)
        df = df.sort_values(['source', 'st', 'County Name', 'place', 'sort'])
        df = df.drop('sort', axis=1)
        df['file'] = table
        df_meta = pd.read_excel(os.path.join(path_dict, f'CHAS data dictionary {years}.xlsx'), sheet_name='All Tables')
        cols = list(df_meta.columns)
        cols[1] = 'Estimate'
        df_meta.columns = cols
        df = df.merge(df_meta, on='Estimate', how='left')
        list_df_tables.append(df)
    df_tables = pd.concat(list_df_tables)
    list_df_years.append(df_tables)

df_chas = pd.concat(list_df_years)
df_chas = df_chas.reset_index(drop=True)
df_chas

In [ ]:
df_codes = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name='CDPcodes')
df_codes = df_codes[df_codes['MPO'] == 'SACOG']
df_codes['place'] = df_codes['place'].astype(str).apply('{:0>5}'.format)
df_codes['state'] = df_codes['state'].astype(str).apply('{:0>2}'.format)

def split_est(x):
    return int(x.split('est')[1])


list_df_years = []
for years in os.listdir(path_data):
    print(''); print(''); print(years)
    list_df_tables = []
    for table in tqdm(os.listdir(os.path.join(path_data, years))):
        df = pd.read_csv(os.path.join(path_data, years, table), encoding='latin-1')
        if 'PLACE' in df.columns:
            df = df.rename(columns={'NAME':'name', 'ST':'st', 'PLACE':'place'})
        df['place'] = df['place'].astype(str).apply('{:0>5}'.format)
        df['st'   ] = df['st'   ].astype(str).apply('{:0>2}'.format)
        df = df.drop(['sumlevel', 'geoid'], axis=1)
        df = df[(df['place'].isin(df_codes['place'].unique())) & (df['st'].isin(df_codes['state'].unique()))]
        df = df.merge(df_codes[['place', 'County Name']], on='place', how='left')
        df.loc[df['place'].isin(df_codes[df_codes['Incorporated'] != 'Yes']['place'].unique()), 'name' ] = 'Unincorporated'
        df.loc[df['place'].isin(df_codes[df_codes['Incorporated'] != 'Yes']['place'].unique()), 'place'] = 'Unincorporated'
        df = df.groupby(['source', 'st', 'County Name', 'place', 'name'], as_index=False).sum()
        df = df.melt(id_vars = ['source', 'st', 'County Name', 'place', 'name'], var_name = 'Estimate', value_name = 'Households')
        df = df[~df['Estimate'].str.contains('moe')] # Need to roll up MOE separately
        df['sort'] = df['Estimate'].apply(split_est)
        df = df.sort_values(['source', 'st', 'County Name', 'place', 'sort'])
        df = df.drop('sort', axis=1)
        df['file'] = table
        tab_name = table.replace('Table', 'Table ', regex=True).replace('.csv', '', regex=True)
        df_meta = pd.read_excel(os.path.join(path_dict, f'CHAS data dictionary {years}.xlsx'), sheet_name=tab_name)
        cols = list(df_meta.columns)
        cols[0] = 'Estimate'
        df_meta.columns = cols
        df = df.merge(df_meta, on='Estimate', how='left')
        list_df_tables.append(df)
    df_tables = pd.concat(list_df_tables)
    list_df_years.append(df_tables)

df_chas = pd.concat(list_df_years)
df_chas = df_chas.reset_index(drop=True)
df_chas

In [ ]:
### RHNA Indicators ------

## POPEMP-21 Household income level by tenure
# Table7
# Household type == All
# Cost burden == All
popemp_21 = [
  'T7_est3'
, 'T7_est24'
, 'T7_est45'
, 'T7_est66'
, 'T7_est87'
, 'T7_est109'
, 'T7_est130'
, 'T7_est151'
, 'T7_est172'
, 'T7_est193'
]

## OVER-01 Overcrowding by tenure and severity
# Table10
# Household income == All
# Family type == All
over_1 = [
  'T10_est3'
, 'T10_est24'
, 'T10_est45'
, 'T10_est67'
, 'T10_est88'
, 'T10_est109'
]

## OVER-02 Overcrowding severity
# Table10
# combine renters and owners
# Household income == All
# Family type == All
over_2 = [
  'T10_est3'
, 'T10_est24'
, 'T10_est45'
, 'T10_est67'
, 'T10_est88'
, 'T10_est109'
]

## OVER-04 Overcrowding by income level
# Table10
# combine renters and owners
# Family type == All
over_4 = [
  'T10_est4'
, 'T10_est8'
, 'T10_est12'
, 'T10_est16'
, 'T10_est20'
, 'T10_est25'
, 'T10_est29'
, 'T10_est33'
, 'T10_est37'
, 'T10_est41'
, 'T10_est46'
, 'T10_est50'
, 'T10_est54'
, 'T10_est58'
, 'T10_est62'
, 'T10_est68'
, 'T10_est72'
, 'T10_est76'
, 'T10_est80'
, 'T10_est84'
, 'T10_est89'
, 'T10_est93'
, 'T10_est97'
, 'T10_est101'
, 'T10_est105'
, 'T10_est110'
, 'T10_est114'
, 'T10_est118'
, 'T10_est122'
, 'T10_est126'
]

## OVER-05 Cost burden by income level
# Table8
# combine renters and owners
# Facilities == All
over_5 = [
  'T8_est4'
, 'T8_est7'
, 'T8_est10'
, 'T8_est13'
, 'T8_est17'
, 'T8_est20'
, 'T8_est23'
, 'T8_est26'
, 'T8_est30'
, 'T8_est33'
, 'T8_est36'
, 'T8_est39'
, 'T8_est43'
, 'T8_est46'
, 'T8_est49'
, 'T8_est52'
, 'T8_est56'
, 'T8_est59'
, 'T8_est62'
, 'T8_est65'
, 'T8_est70'
, 'T8_est73'
, 'T8_est76'
, 'T8_est79'
, 'T8_est83'
, 'T8_est86'
, 'T8_est89'
, 'T8_est92'
, 'T8_est96'
, 'T8_est99'
, 'T8_est102'
, 'T8_est105'
, 'T8_est109'
, 'T8_est112'
, 'T8_est115'
, 'T8_est118'
, 'T8_est122'
, 'T8_est125'
, 'T8_est128'
, 'T8_est131'
]

## OVER-08 Cost burden by race/ethnicity
# Table9
# combine renters and owners
over_8 = [
    'T9_est4'
    , 'T9_est5'
    , 'T9_est6'
    , 'T9_est7'
    , 'T9_est9'
    , 'T9_est10'
    , 'T9_est11'
    , 'T9_est12'
    , 'T9_est14'
    , 'T9_est15'
    , 'T9_est16'
    , 'T9_est17'
    , 'T9_est19'
    , 'T9_est20'
    , 'T9_est21'
    , 'T9_est22'
    , 'T9_est24'
    , 'T9_est25'
    , 'T9_est26'
    , 'T9_est27'
    , 'T9_est29'
    , 'T9_est30'
    , 'T9_est31'
    , 'T9_est32'
    , 'T9_est34'
    , 'T9_est35'
    , 'T9_est36'
    , 'T9_est37'
    , 'T9_est40'
    , 'T9_est41'
    , 'T9_est42'
    , 'T9_est43'
    , 'T9_est45'
    , 'T9_est46'
    , 'T9_est47'
    , 'T9_est48'
    , 'T9_est50'
    , 'T9_est51'
    , 'T9_est52'
    , 'T9_est53'
    , 'T9_est55'
    , 'T9_est56'
    , 'T9_est57'
    , 'T9_est58'
    , 'T9_est60'
    , 'T9_est61'
    , 'T9_est62'
    , 'T9_est63'
    , 'T9_est65'
    , 'T9_est66'
    , 'T9_est67'
    , 'T9_est68'
    , 'T9_est70'
    , 'T9_est71'
    , 'T9_est72'
    , 'T9_est73'
]

## OVER-09 Cost burden by household size
# Table7
# combine renters and owners
# large families vs all other households (roll up)
over_9 = [
    'T7_est5'
    , 'T7_est6'
    , 'T7_est7'
    , 'T7_est9'
    , 'T7_est10'
    , 'T7_est11'
    , 'T7_est13'
    , 'T7_est14'
    , 'T7_est15'
    , 'T7_est17'
    , 'T7_est18'
    , 'T7_est19'
    , 'T7_est21'
    , 'T7_est22'
    , 'T7_est23'
    , 'T7_est26'
    , 'T7_est27'
    , 'T7_est28'
    , 'T7_est30'
    , 'T7_est31'
    , 'T7_est32'
    , 'T7_est34'
    , 'T7_est35'
    , 'T7_est36'
    , 'T7_est38'
    , 'T7_est39'
    , 'T7_est40'
    , 'T7_est42'
    , 'T7_est43'
    , 'T7_est44'
    , 'T7_est47'
    , 'T7_est48'
    , 'T7_est49'
    , 'T7_est51'
    , 'T7_est52'
    , 'T7_est53'
    , 'T7_est55'
    , 'T7_est56'
    , 'T7_est57'
    , 'T7_est59'
    , 'T7_est60'
    , 'T7_est61'
    , 'T7_est63'
    , 'T7_est64'
    , 'T7_est65'
    , 'T7_est68'
    , 'T7_est69'
    , 'T7_est70'
    , 'T7_est72'
    , 'T7_est73'
    , 'T7_est74'
    , 'T7_est76'
    , 'T7_est77'
    , 'T7_est78'
    , 'T7_est80'
    , 'T7_est81'
    , 'T7_est82'
    , 'T7_est84'
    , 'T7_est85'
    , 'T7_est86'
    , 'T7_est89'
    , 'T7_est90'
    , 'T7_est91'
    , 'T7_est93'
    , 'T7_est94'
    , 'T7_est95'
    , 'T7_est97'
    , 'T7_est98'
    , 'T7_est99'
    , 'T7_est101'
    , 'T7_est102'
    , 'T7_est103'
    , 'T7_est105'
    , 'T7_est106'
    , 'T7_est107'
    , 'T7_est111'
    , 'T7_est112'
    , 'T7_est113'
    , 'T7_est115'
    , 'T7_est116'
    , 'T7_est117'
    , 'T7_est119'
    , 'T7_est120'
    , 'T7_est121'
    , 'T7_est123'
    , 'T7_est124'
    , 'T7_est125'
    , 'T7_est127'
    , 'T7_est128'
    , 'T7_est129'
    , 'T7_est132'
    , 'T7_est133'
    , 'T7_est134'
    , 'T7_est136'
    , 'T7_est137'
    , 'T7_est138'
    , 'T7_est140'
    , 'T7_est141'
    , 'T7_est142'
    , 'T7_est144'
    , 'T7_est145'
    , 'T7_est146'
    , 'T7_est148'
    , 'T7_est149'
    , 'T7_est150'
    , 'T7_est153'
    , 'T7_est154'
    , 'T7_est155'
    , 'T7_est157'
    , 'T7_est158'
    , 'T7_est159'
    , 'T7_est161'
    , 'T7_est162'
    , 'T7_est163'
    , 'T7_est165'
    , 'T7_est166'
    , 'T7_est167'
    , 'T7_est169'
    , 'T7_est170'
    , 'T7_est171'
    , 'T7_est174'
    , 'T7_est175'
    , 'T7_est176'
    , 'T7_est178'
    , 'T7_est179'
    , 'T7_est180'
    , 'T7_est182'
    , 'T7_est183'
    , 'T7_est184'
    , 'T7_est186'
    , 'T7_est187'
    , 'T7_est188'
    , 'T7_est190'
    , 'T7_est191'
    , 'T7_est192'
    , 'T7_est195'
    , 'T7_est196'
    , 'T7_est197'
    , 'T7_est199'
    , 'T7_est200'
    , 'T7_est201'
    , 'T7_est203'
    , 'T7_est204'
    , 'T7_est205'
    , 'T7_est207'
    , 'T7_est208'
    , 'T7_est209'
    , 'T7_est211'
    , 'T7_est212'
    , 'T7_est213'
]

## LGFEM-03 Household size by household income level
# Table16
# combine renters and owners
# large families vs all other households (subtract All - large)
lgfem_3 = [
    'T16_est3'
    , 'T16_est12'
    , 'T16_est24'
    , 'T16_est33'
    , 'T16_est45'
    , 'T16_est54'
    , 'T16_est66'
    , 'T16_est75'
    , 'T16_est88'
    , 'T16_est97'
    , 'T16_est109'
    , 'T16_est118'
    , 'T16_est130'
    , 'T16_est139'
    , 'T16_est151'
    , 'T16_est160'
]

## SEN-01 Senior households by income and tenure
# Table7
# cost burden == All
# household type == elderly non family
sen_1 = [
      'T7_est16'
    , 'T7_est37'
    , 'T7_est58'
    , 'T7_est79'
    , 'T7_est100'
    , 'T7_est122'
    , 'T7_est143'
    , 'T7_est164'
    , 'T7_est185'
    , 'T7_est206'

]

## SEN-02 Cost-burdened senior households by income level
# Table7
# Cost burden != All
# household type == elderly non family
sen_3 = [
      'T7_est17'
    , 'T7_est18'
    , 'T7_est19'
    , 'T7_est38'
    , 'T7_est39'
    , 'T7_est40'
    , 'T7_est59'
    , 'T7_est60'
    , 'T7_est61'
    , 'T7_est80'
    , 'T7_est81'
    , 'T7_est82'
    , 'T7_est101'
    , 'T7_est102'
    , 'T7_est103'
    , 'T7_est123'
    , 'T7_est124'
    , 'T7_est125'
    , 'T7_est144'
    , 'T7_est145'
    , 'T7_est146'
    , 'T7_est165'
    , 'T7_est166'
    , 'T7_est167'
    , 'T7_est186'
    , 'T7_est187'
    , 'T7_est188'
    , 'T7_est207'
    , 'T7_est208'
    , 'T7_est209'
]

## ELI-01 Households by household income level
# Table7
# household type == ALl
# Cost burden == All
# combine renters and owners
eli_1 = [
    'T7_est3'
    , 'T7_est24'
    , 'T7_est45'
    , 'T7_est66'
    , 'T7_est87'
    , 'T7_est109'
    , 'T7_est130'
    , 'T7_est151'
    , 'T7_est172'
    , 'T7_est193'
]

## ELI-02 Household income distribution by race
# Table1
# roll up to race/ethnicity and household income categories
eli_2 = [
    'T1_est5'
    , 'T1_est6'
    , 'T1_est7'
    , 'T1_est8'
    , 'T1_est9'
    , 'T1_est10'
    , 'T1_est12'
    , 'T1_est13'
    , 'T1_est14'
    , 'T1_est15'
    , 'T1_est16'
    , 'T1_est17'
    , 'T1_est19'
    , 'T1_est20'
    , 'T1_est21'
    , 'T1_est22'
    , 'T1_est23'
    , 'T1_est24'
    , 'T1_est26'
    , 'T1_est27'
    , 'T1_est28'
    , 'T1_est29'
    , 'T1_est30'
    , 'T1_est31'
    , 'T1_est33'
    , 'T1_est34'
    , 'T1_est35'
    , 'T1_est36'
    , 'T1_est37'
    , 'T1_est38'
    , 'T1_est41'
    , 'T1_est42'
    , 'T1_est43'
    , 'T1_est44'
    , 'T1_est45'
    , 'T1_est46'
    , 'T1_est48'
    , 'T1_est49'
    , 'T1_est50'
    , 'T1_est51'
    , 'T1_est52'
    , 'T1_est53'
    , 'T1_est55'
    , 'T1_est56'
    , 'T1_est57'
    , 'T1_est58'
    , 'T1_est59'
    , 'T1_est60'
    , 'T1_est62'
    , 'T1_est63'
    , 'T1_est64'
    , 'T1_est65'
    , 'T1_est66'
    , 'T1_est67'
    , 'T1_est69'
    , 'T1_est70'
    , 'T1_est71'
    , 'T1_est72'
    , 'T1_est73'
    , 'T1_est74'
    , 'T1_est78'
    , 'T1_est79'
    , 'T1_est80'
    , 'T1_est81'
    , 'T1_est82'
    , 'T1_est83'
    , 'T1_est85'
    , 'T1_est86'
    , 'T1_est87'
    , 'T1_est88'
    , 'T1_est89'
    , 'T1_est90'
    , 'T1_est92'
    , 'T1_est93'
    , 'T1_est94'
    , 'T1_est95'
    , 'T1_est96'
    , 'T1_est97'
    , 'T1_est99'
    , 'T1_est100'
    , 'T1_est101'
    , 'T1_est102'
    , 'T1_est103'
    , 'T1_est104'
    , 'T1_est106'
    , 'T1_est107'
    , 'T1_est108'
    , 'T1_est109'
    , 'T1_est110'
    , 'T1_est111'
    , 'T1_est114'
    , 'T1_est115'
    , 'T1_est116'
    , 'T1_est117'
    , 'T1_est118'
    , 'T1_est119'
    , 'T1_est121'
    , 'T1_est122'
    , 'T1_est123'
    , 'T1_est124'
    , 'T1_est125'
    , 'T1_est126'
    , 'T1_est128'
    , 'T1_est129'
    , 'T1_est130'
    , 'T1_est131'
    , 'T1_est132'
    , 'T1_est133'
    , 'T1_est135'
    , 'T1_est136'
    , 'T1_est137'
    , 'T1_est138'
    , 'T1_est139'
    , 'T1_est140'
    , 'T1_est142'
    , 'T1_est143'
    , 'T1_est144'
    , 'T1_est145'
    , 'T1_est146'
    , 'T1_est147'
]


***

POPEMP-21

***

In [ ]:
## POPEMP-21 Household income level by tenure
# Table7
# Household type == All
# Cost burden == All
popemp_21 = [
  'T7_est3'
, 'T7_est24'
, 'T7_est45'
, 'T7_est66'
, 'T7_est87'
, 'T7_est109'
, 'T7_est130'
, 'T7_est151'
, 'T7_est172'
, 'T7_est193'
]

df_ind = df_chas[df_chas['Estimate'].isin(popemp_21)]
df_ind = df_ind.reset_index(drop=True)

df_ind.head()